# RI #1: Retrieval levers and their costs

This notebook turns the retrieval knob on the harness from notebook 01. It scores each retrieval lever on the same contract questions, and puts what the lever gains next to what it costs in CPU time per question and index size.

By the end, you can:

- Fuse BM25 and dense rankings with weighted reciprocal rank fusion (RRF), and choose the dense weight by measuring it.
- Compare SPLADE and ColBERT late interaction with the BM25, dense, and hybrid first stages.
- Rerank the best first stage with two cross-encoders and with ColBERT, and change the rerank depth without a model.
- Read failure rate, MRR, latency, and index size together, and name which lever wins on which metric, on which corpus.
- Measure exact search, HNSW, and 1-bit and 2-bit quantization on a Qdrant server, and see why they barely differ at this size.

The surprising result comes first. SPLADE alone, a learned sparse model that needs no fusion and no reranker, beats the tuned BM25 and dense hybrid on all three slices and on both metrics. On the full CUAD split its failure rate is 0.2066 against the hybrid's 0.2436, 46 fewer questions of 1,244 with no gold chunk in the top 10, and its MRR is 0.5687 against 0.4969. On CUAD, every reranker measured here, including the 568M-parameter `bge-reranker-v2-m3`, lowers SPLADE's MRR. Section 7 names the winner for each metric and the slice it wins on.

## 1. Setup and the three slices

Run the cells in order with `RUN_EXPENSIVE=False`, as in notebook 01. The default path needs no API key, no embedding model, and no Qdrant server. BM25 runs live. Every other arm is read from the cache in `outputs/baseline_cache`, next to notebook 01's measurements,, which stores each arm's ranked chunks to depth 50 and each reranker's score for every chunk of a question's contract. The fusion, reranking, and scoring cells compute from those stored rankings, so you can change a fusion weight, a first stage, or a rerank depth and see the result in seconds without a model. The notebook makes no language-model calls, and every reranker runs locally.

Enabling `RUN_EXPENSIVE` re-embeds every slice, re-scores every candidate with all three rerankers, and re-times each arm under an 8-CPU limit with the GPU hidden, which stands in for the bootcamp container. On 8 CPUs that takes a few hours, most of it `bge-reranker-v2-m3`.

The comparison uses three slices. Every question is scoped to its own contract, as in notebook 01:

- CUAD 20: the 20 CUAD test contracts and 250 answerable questions from notebook 01, 954 chunks of 1,200 characters with 200 characters of overlap.
- ContractNLI 20: the first 20 ContractNLI test NDAs from notebook 01's answering section, 179 answerable questions over 214 chunks.
- CUAD all: the full CUAD test split, 102 contracts, 4,811 chunks, and 1,244 answerable questions. It is the slice the earlier hybrid experiment tuned its weight on, and the largest question set here, so the fusion weight is chosen on it. The rerankers do not run on it, because scoring 60,000 pairs with `bge-reranker-v2-m3` on 8 CPUs takes about ten hours.

In [1]:
import hashlib
import json
import os
import statistics
import tempfile
import time
import urllib.request
from importlib.metadata import version
from pathlib import Path

from aieng.advanced_rag.cache import CacheKey, TieredResultCache
from aieng.advanced_rag.cache.directories import store_directory
from aieng.advanced_rag.corpora import contract_nli, cuad
from aieng.advanced_rag.evaluation.metrics import score_retrieval
from aieng.advanced_rag.retrieval.retrievers import run_retriever
from aieng.advanced_rag.retrieval.sparse import BM25Retriever, HybridRetriever, weighted_rrf
from aieng.advanced_rag.retrieval.stores import QdrantSettings, QdrantStore, export_collection, fetch_collection


# Embedding, reranking, and latency measurement ship off. Enable the flag to
# refresh the measured artifacts; the default path reads them through the
# shared result cache and needs no model, no API key, and no Qdrant server.
RUN_EXPENSIVE = False

K = 10
DEPTH = 50
CPUS = 8
DENSE_MODEL = "BAAI/bge-small-en-v1.5"

if RUN_EXPENSIVE:
    # The latency figures are for the bootcamp container: 8 CPUs, no GPU. Both
    # settings have to land before torch or onnxruntime is imported.
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    os.environ["OMP_NUM_THREADS"] = str(CPUS)
    os.environ["MKL_NUM_THREADS"] = str(CPUS)
    os.sched_setaffinity(0, set(sorted(os.sched_getaffinity(0))[:CPUS]))

REPO = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "aieng-advanced-rag").is_dir())
RI = REPO / "implementations" / "regulatory_policy_qa"
cache = TieredResultCache(global_dir=RI / "outputs" / "baseline_cache")


def measured_result(key, compute):
    """Read a measured artifact or explicitly refresh it."""
    if RUN_EXPENSIVE:
        return cache.load_or_compute(key, compute, force_refresh=True, write_tier="global").value
    cached = cache.get(key)
    if cached is None:
        raise FileNotFoundError("Missing measured artifact; enable RUN_EXPENSIVE to create it")
    return cached.value


cuad_path = cuad.fetch_cuad_test()
nli_path = contract_nli.fetch_contract_nli()
SLICES = {
    "CUAD 20": (cuad_path, cuad.build_slice(cuad_path, max_documents=20), 20),
    "ContractNLI 20": (nli_path, contract_nli.build_slice(nli_path, max_documents=20), 20),
    "CUAD all": (cuad_path, cuad.build_slice(cuad_path), None),
}
DIGESTS = {path: hashlib.sha256(path.read_bytes()).hexdigest() for path in (cuad_path, nli_path)}
for name, (_, prepared, _) in SLICES.items():
    summary = prepared.summary()
    per_doc = summary["chunks"] / summary["contracts"]
    print(
        f"{name:<15}{summary['contracts']:>4} contracts {summary['chunks']:>5} chunks "
        f"({per_doc:.1f} per contract)  {summary['answerable_queries']:>5} answerable questions"
    )
assert SLICES["CUAD 20"][1].summary()["answerable_queries"] == 250
assert SLICES["ContractNLI 20"][1].summary()["answerable_queries"] == 179
assert SLICES["CUAD all"][1].summary()["answerable_queries"] == 1244

CUAD 20          20 contracts   954 chunks (47.7 per contract)    250 answerable questions
ContractNLI 20   20 contracts   214 chunks (10.7 per contract)    179 answerable questions
CUAD all        102 contracts  4811 chunks (47.2 per contract)   1244 answerable questions


## 2. The scores and the costs

Retrieval failure and MRR are the notebook 01 definitions, computed on answerable questions only. Failure is the share of questions with no gold chunk in the top `k`; MRR is the mean of `1 / rank` of the first gold chunk in the top 10.

ContractNLI needs one change. Its NDAs average 10.7 chunks, so the top 10 holds nearly the whole document, and failure at 10 is close to zero for every arm. ContractNLI rows report failure at `k=3` instead, and MRR as before.

Two costs sit beside the scores:

- Latency is the median milliseconds per question on 8 CPUs, measured end to end: encoding the question, searching the contract, and returning the top 10. A reranker adds the time to score its candidates, which is the depth divided by its measured pairs per second. The host was shared with other jobs during the cached run, so the notebook prints the load average next to the figures, and figures measured with `scripts/measure_cpu_latency.py` on an idle host beside them where they cover an arm.
- Index size is the bytes of the representation each arm searches: 4 bytes per dimension for dense vectors, 8 bytes per stored term for BM25 and SPLADE sparse vectors, and 4 bytes per dimension per token for ColBERT. It leaves out the model weights and any store overhead.

The cell below scores BM25 live on all three slices. The first assertion reproduces notebook 01's CUAD 20 figures, and the second checks the full-split figure.

In [2]:
# ContractNLI documents average 10.7 chunks, so a top 10 holds nearly the whole
# document and failure at 10 says almost nothing. Its rows also report k=3.
SHALLOW_K = {"CUAD 20": K, "ContractNLI 20": 3, "CUAD all": K}


def score(runs, slice_name, k=K):
    """Return failure at `k` and MRR in the top 10 for ranked chunk IDs."""
    qrels = SLICES[slice_name][1].qrels
    report = score_retrieval({q: ids[:k] for q, ids in runs.items()}, qrels, k=k)
    mrr = score_retrieval({q: ids[:K] for q, ids in runs.items()}, qrels, k=K).mrr
    return report.retrieval_failure_rate, mrr


def slice_config(slice_name):
    """Name every input that fixes a slice's chunks and questions."""
    path, _, max_documents = SLICES[slice_name]
    return {
        "corpus": "cuad" if path == cuad_path else "contract-nli",
        "corpus_sha256": DIGESTS[path],
        "max_documents": max_documents,
        "chunk_chars": 1200,
        "overlap_chars": 200,
        "scope": "contract",
    }


def as_indices(runs, slice_name):
    """Store rankings as positions in the slice's chunk list."""
    position = {chunk.chunk_id: i for i, chunk in enumerate(SLICES[slice_name][1].chunks)}
    return {q: [position[c] for c in ids] for q, ids in runs.items()}


def as_ids(indexed, slice_name):
    """Turn cached positions back into chunk identifiers."""
    chunks = SLICES[slice_name][1].chunks
    return {q: [chunks[i].chunk_id for i in positions] for q, positions in indexed.items()}


def fuse(arms, weights, depth=DEPTH, limit=K):
    """Fuse cached rankings with weighted RRF, as HybridRetriever.search does."""
    first = arms[0]
    return {q: [c for c, _ in weighted_rrf([arm[q][:depth] for arm in arms], weights, limit=limit)] for q in first}


# BM25 needs no model, so it runs live on every slice, as in notebook 01.
rankings = {name: {} for name in SLICES}
for name, (_, prepared, _) in SLICES.items():
    lexical = BM25Retriever()
    lexical.index(prepared.chunks)
    rankings[name]["bm25"] = run_retriever(lexical, prepared.answerable, k=DEPTH)

assert score(rankings["CUAD 20"]["bm25"], "CUAD 20") == (0.2600, 0.4528)
assert tuple(round(v, 4) for v in score(rankings["CUAD all"]["bm25"], "CUAD all")) == (0.2548, 0.4637)
for name in SLICES:
    failure, mrr = score(rankings[name]["bm25"], name, SHALLOW_K[name])
    print(f"BM25 on {name:<15} failure@{SHALLOW_K[name]:<3}{failure:.4f}  MRR {mrr:.4f}")

BM25 on CUAD 20         failure@10 0.2600  MRR 0.4528
BM25 on ContractNLI 20  failure@3  0.1508  MRR 0.7395
BM25 on CUAD all        failure@10 0.2548  MRR 0.4637


## 3. Five single arms

The dense arm is `BAAI/bge-small-en-v1.5` with its published query prompt, the shipped default from notebook 01, searched through the package's Qdrant store in process. `dense, no prompt` is the same model without the prompt, which the earlier full-split experiment used; it appears only on CUAD all, to compare against that experiment. SPLADE (`prithivida/Splade_PP_en_v1`) expands each chunk into weighted vocabulary terms. ColBERT (`answerdotai/answerai-colbert-small-v1`) keeps a 96-dimension vector per token and scores a chunk by MaxSim: for each question token, the best match among the chunk's tokens, summed. All four models run locally on CPU through `sentence-transformers` or `fastembed`.

The cell asserts that the dense arm still reproduces notebook 01's 0.2400 failure and 0.4632 MRR.

In [3]:
ARM_MODELS = {
    "dense": DENSE_MODEL,
    "dense, no prompt": DENSE_MODEL,
    "splade": "prithivida/Splade_PP_en_v1",
    "colbert": "answerdotai/answerai-colbert-small-v1",
}
LATENCY_QUERIES = 50

if RUN_EXPENSIVE:
    from aieng.advanced_rag.retrieval.embedders import SentenceTransformerEmbedder
    from aieng.advanced_rag.retrieval.representations import ColbertRetriever, SpladeRetriever
    from aieng.advanced_rag.retrieval.retrievers import DenseRetriever


def build_arm(arm):
    """Return an unindexed retriever for one arm."""
    if arm == "bm25":
        return BM25Retriever()
    if arm == "dense":
        return DenseRetriever(SentenceTransformerEmbedder(DENSE_MODEL))
    if arm == "dense, no prompt":
        return DenseRetriever(SentenceTransformerEmbedder(DENSE_MODEL, query_prompt=""))
    if arm == "splade":
        return SpladeRetriever()
    if arm == "colbert":
        return ColbertRetriever()
    raise ValueError(arm)


def index_bytes(retriever, arm):
    """Return the bytes of the representation an arm searches."""
    if arm in ("bm25", "splade"):
        # A sparse vector stores a 4-byte index and a 4-byte weight per term.
        return 8 * sum(len(vector) for vector in retriever._vectors.values())
    if arm == "colbert":
        return sum(matrix.nbytes for matrix in retriever._tokens.values())
    return 4 * len(retriever.vectors) * retriever.vectors.dimension


def search_latency(retriever, queries):
    """Return per-query milliseconds for top-10 search, query encoding included."""
    for query in queries[:5]:
        retriever.search(query, k=K)
    times = []
    for query in queries[:LATENCY_QUERIES]:
        began = time.perf_counter()
        retriever.search(query, k=K)
        times.append(1000 * (time.perf_counter() - began))
    return {"median_ms": round(statistics.median(times), 2), "mean_ms": round(statistics.mean(times), 2)}


def arm_key(arm, slice_name):
    """Name every input that fixes one arm's rankings."""
    return CacheKey(
        "ri1",
        "lever-rankings",
        ARM_MODELS.get(arm, "bm25").split("/")[-1],
        {**slice_config(slice_name), "arm": arm, "model": ARM_MODELS.get(arm), "depth": DEPTH},
    )


def measure_arm(arm, slice_name):
    """Index one arm, rank every answerable question to depth 50, record costs."""
    prepared = SLICES[slice_name][1]
    retriever = build_arm(arm)
    began = time.perf_counter()
    retriever.index(prepared.chunks)
    indexing_seconds = time.perf_counter() - began
    result = {
        "runs": as_indices(run_retriever(retriever, prepared.answerable, k=DEPTH), slice_name),
        "index_bytes": index_bytes(retriever, arm),
        "index_seconds": round(indexing_seconds, 1),
        "versions": {name: version(name) for name in ("fastembed", "sentence-transformers", "qdrant-client")},
    }
    if slice_name == "CUAD 20":
        result["latency"] = {**search_latency(retriever, prepared.answerable), "load_average": os.getloadavg()[0]}
    return result


arm_results = {name: {} for name in SLICES}
for name in SLICES:
    for arm in ARM_MODELS:
        if arm == "dense, no prompt" and name != "CUAD all":
            continue
        measured = measured_result(arm_key(arm, name), lambda arm=arm, name=name: measure_arm(arm, name))
        arm_results[name][arm] = measured
        rankings[name][arm] = as_ids(measured["runs"], name)

# BM25's costs are measured live on CUAD 20: it is the cheapest arm to time.
bm25_live = BM25Retriever()
bm25_live.index(SLICES["CUAD 20"][1].chunks)
arm_results["CUAD 20"]["bm25"] = {
    "index_bytes": index_bytes(bm25_live, "bm25"),
    "latency": search_latency(bm25_live, SLICES["CUAD 20"][1].answerable),
}

# The dense arm reproduces notebook 01 and the published guards to four decimals.
assert tuple(round(v, 4) for v in score(rankings["CUAD 20"]["dense"], "CUAD 20")) == (0.2400, 0.4632)
source = "recomputed" if RUN_EXPENSIVE else "committed measurement"
print(f"single arms, top {K}, scoped to the question's contract ({source})")
print(f"{'arm':<18}" + "".join(f"{name + ' fail':>20}{'MRR':>8}" for name in SLICES))
for arm in ("bm25", "dense", "dense, no prompt", "splade", "colbert"):
    cells = []
    for name in SLICES:
        if arm in rankings[name]:
            failure, mrr = score(rankings[name][arm], name, SHALLOW_K[name])
            cells.append(f"{failure:>20.4f}{mrr:>8.4f}")
        else:
            cells.append(f"{'':>20}{'':>8}")
    print(f"{arm:<18}" + "".join(cells))
print("ContractNLI failure is at k=3; CUAD failure is at k=10")

single arms, top 10, scoped to the question's contract (committed measurement)
arm                       CUAD 20 fail     MRR ContractNLI 20 fail     MRR       CUAD all fail     MRR
bm25                            0.2600  0.4528              0.1508  0.7395              0.2548  0.4637
dense                           0.2400  0.4632              0.1732  0.6893              0.3063  0.4156
dense, no prompt                                                                        0.3063  0.3995
splade                          0.2000  0.6139              0.0838  0.8180              0.2066  0.5687
colbert                         0.2240  0.5348              0.0894  0.7706              0.2669  0.4972
ContractNLI failure is at k=3; CUAD failure is at k=10


## 4. Hybrid fusion and the dense weight

`HybridRetriever` searches each arm to depth 50 and fuses the two rankings with weighted RRF: a chunk at rank `r` in an arm earns `weight / (60 + r)` from it. The dense arm gets `dense_weight` and BM25 gets the rest. The cell replays that computation over the cached rankings for 21 weights, from BM25 alone (0.00) to dense alone (1.00).

In [4]:
WEIGHTS = [round(0.05 * i, 2) for i in range(21)]
SHIPPED_WEIGHT = HybridRetriever.__init__.__defaults__[0]


def sweep(slice_name, dense_arm="dense", lexical_arm="bm25"):
    """Return (weight, failure, MRR) for weighted RRF of two cached arms."""
    arms = [rankings[slice_name][dense_arm], rankings[slice_name][lexical_arm]]
    return [(w, *score(fuse(arms, [w, 1 - w]), slice_name, SHALLOW_K[slice_name])) for w in WEIGHTS]


sweeps = {
    "CUAD all": sweep("CUAD all"),
    "CUAD all, no prompt": sweep("CUAD all", "dense, no prompt"),
    "CUAD 20": sweep("CUAD 20"),
    "ContractNLI 20": sweep("ContractNLI 20"),
}
print(f"weighted RRF of dense (BGE prompt unless noted) and BM25; shipped default dense_weight={SHIPPED_WEIGHT}")
print(f"{'dense weight':<13}" + "".join(f"{name:>26}" for name in sweeps))
print(f"{'':<13}" + "".join(f"{'fail':>17}{'MRR':>9}" for _ in sweeps))
for row, weight in enumerate(WEIGHTS):
    marks = "  <- shipped" if weight == SHIPPED_WEIGHT else ""
    line = "".join(f"{sweeps[name][row][1]:>17.4f}{sweeps[name][row][2]:>9.4f}" for name in sweeps)
    print(f"{weight:<13.2f}{line}{marks}")
print("weight 0.00 is BM25 alone and 1.00 is dense alone")
for name, rows in sweeps.items():
    best_failure = min(rows, key=lambda row: (row[1], -row[2]))
    best_mrr = max(rows, key=lambda row: (row[2], -row[1]))
    print(
        f"{name:<22} lowest failure at {best_failure[0]:.2f} ({best_failure[1]:.4f}); highest MRR at {best_mrr[0]:.2f} ({best_mrr[2]:.4f})"
    )

weighted RRF of dense (BGE prompt unless noted) and BM25; shipped default dense_weight=0.2
dense weight                   CUAD all       CUAD all, no prompt                   CUAD 20            ContractNLI 20
                          fail      MRR             fail      MRR             fail      MRR             fail      MRR
0.00                    0.2548   0.4637           0.2548   0.4637           0.2600   0.4528           0.1508   0.7395
0.05                    0.2516   0.4700           0.2540   0.4704           0.2600   0.4595           0.1508   0.7395
0.10                    0.2508   0.4835           0.2516   0.4822           0.2480   0.4743           0.1453   0.7437
0.15                    0.2468   0.4910           0.2516   0.4888           0.2480   0.4865           0.1508   0.7474
0.20                    0.2436   0.4969           0.2460   0.4955           0.2320   0.4967           0.1397   0.7533  <- shipped
0.25                    0.2468   0.4989           0.2508   0.4942      

On the full CUAD split the lowest failure is at a dense weight of 0.20: 0.2436 with the BGE query prompt, against 0.2629 at equal weight and 0.2548 for BM25 alone. Without the prompt the minimum is also at 0.20, at 0.2460. The gap from equal weight is 24 of 1,244 questions. MRR peaks at 0.25 (0.4989), with 0.20 at 0.4969. `HybridRetriever` therefore defaults to `dense_weight=0.2`, which the `<- shipped` marker shows.

The two 20-document slices put the optimum higher. On CUAD 20, MRR peaks at 0.40 (0.5294 against 0.4967 at 0.20), and on ContractNLI 20 at 0.40 as well (0.7660 against 0.7533). Their failure minima, at 0.85 and 0.65, sit on curves that move by one or two questions from one weight to the next, so they carry little information. The weight was chosen on the slice with five times the questions. On your corpus, run the sweep before trusting the default: the cell needs only the two cached rankings, so a participant can rerun it with `WEIGHTS` changed in seconds.

The equal-weight figure is the reason a sweep matters. On the full split, unweighted fusion (0.2629) loses to BM25 alone (0.2548), so a team that tried hybrid search once at equal weight would conclude it does not help on contracts.

## 5. SPLADE as the first stage, and fusing it with dense

SPLADE alone beat every BM25 and dense combination in section 4, on all three slices and on both metrics. The cell asks whether fusing dense into SPLADE helps further, with the same weighted RRF.

On CUAD 20, a little dense helps failure: 0.1840 at a dense weight of 0.3, against 0.2000 for SPLADE alone, at a cost in MRR (0.5942 against 0.6139). On CUAD all and ContractNLI 20 the fusion buys at most two questions, and no weight raises MRR above SPLADE alone. SPLADE alone is the simpler first stage. The 0.3 fusion stays in section 6 as the first stage with the lowest CUAD 20 failure, because a reranker can only reorder what the first stage found.

In [5]:
def sweep_pair(slice_name, left, right, weights):
    """Return (weight, failure, MRR) for weighted RRF of two cached arms."""
    arms = [rankings[slice_name][left], rankings[slice_name][right]]
    return [(w, *score(fuse(arms, [w, 1 - w]), slice_name, SHALLOW_K[slice_name])) for w in weights]


PAIR_WEIGHTS = [round(0.1 * i, 1) for i in range(11)]
print("weighted RRF of dense (weight shown) and SPLADE (1 - weight)")
print(f"{'dense weight':<13}" + "".join(f"{name:>26}" for name in SLICES))
pairs = {name: sweep_pair(name, "dense", "splade", PAIR_WEIGHTS) for name in SLICES}
for row, weight in enumerate(PAIR_WEIGHTS):
    print(f"{weight:<13.1f}" + "".join(f"{pairs[name][row][1]:>17.4f}{pairs[name][row][2]:>9.4f}" for name in SLICES))
print("weight 0.0 is SPLADE alone and 1.0 is dense alone")

weighted RRF of dense (weight shown) and SPLADE (1 - weight)
dense weight                    CUAD 20            ContractNLI 20                  CUAD all


0.0                     0.2000   0.6139           0.0838   0.8180           0.2066   0.5687
0.1                     0.2000   0.6129           0.0838   0.8180           0.2074   0.5660
0.2                     0.1880   0.6050           0.0838   0.8123           0.2050   0.5551
0.3                     0.1840   0.5942           0.0894   0.8084           0.2122   0.5444
0.4                     0.1920   0.5818           0.0838   0.7893           0.2219   0.5312
0.5                     0.1920   0.5669           0.1006   0.7644           0.2331   0.5136
0.6                     0.2080   0.5474           0.1173   0.7506           0.2532   0.4920
0.7                     0.2160   0.5249           0.1397   0.7217           0.2685   0.4735
0.8                     0.2240   0.5069           0.1508   0.7036           0.2805   0.4530
0.9                     0.2360   0.4765           0.1676   0.6900           0.2966   0.4295
1.0                     0.2400   0.4632           0.1732   0.6893           0.30

## 6. Reranking the first stage

A reranker rescores the first stage's top candidates and reorders them. Three run here, all locally on CPU:

- `cross-encoder/ms-marco-MiniLM-L-6-v2`, a 22.7M-parameter cross-encoder trained on web search.
- `BAAI/bge-reranker-v2-m3`, a 568M-parameter multilingual cross-encoder.
- ColBERT MaxSim, which reuses the ColBERT model from section 3 to rescore candidates instead of retrieving them.

The cached artifact holds each reranker's score for every chunk of each question's contract, so any first stage at any depth replays without a model. Change `FIRST_STAGES` or `RERANK_DEPTHS` in the second cell to try another combination. A CUAD contract averages 47.7 chunks, so a depth of 30 already reranks most of it, and the first stage then matters less than the reranker.

In [6]:
RERANKERS = {
    "MiniLM-L6": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "bge-reranker-v2-m3": "BAAI/bge-reranker-v2-m3",
    "ColBERT MaxSim": "answerdotai/answerai-colbert-small-v1",
}
RERANK_SLICES = ("CUAD 20", "ContractNLI 20")

if RUN_EXPENSIVE:
    from aieng.advanced_rag.retrieval.representations import FastembedColbertEncoder, max_sim
    from aieng.advanced_rag.retrieval.rerankers import CrossEncoderReranker


def contract_chunks(slice_name):
    """Return each contract's chunk IDs in slice order."""
    by_doc = {}
    for chunk in SLICES[slice_name][1].chunks:
        by_doc.setdefault(chunk.doc_id, []).append(chunk.chunk_id)
    return by_doc


def score_every_chunk(label, slice_name):
    """Score every chunk of each question's contract, for replay at any depth."""
    prepared = SLICES[slice_name][1]
    texts = {chunk.chunk_id: chunk.indexed_text for chunk in prepared.chunks}
    by_doc = contract_chunks(slice_name)
    scores = {}
    if label == "ColBERT MaxSim":
        encoder = FastembedColbertEncoder()
        ids = [chunk.chunk_id for chunk in prepared.chunks]
        tokens = dict(zip(ids, encoder.encode_documents([texts[i] for i in ids])))
        encode_ms, pairs, pair_seconds = [], 0, 0.0
        for query in prepared.answerable:
            began = time.perf_counter()
            query_tokens = encoder.encode_query(query.question)
            encoded = time.perf_counter()
            scores[query.query_id] = [round(max_sim(query_tokens, tokens[c]), 5) for c in by_doc[query.doc_id]]
            encode_ms.append(1000 * (encoded - began))
            pair_seconds += time.perf_counter() - encoded
            pairs += len(by_doc[query.doc_id])
        timing = {
            "query_encode_ms": round(statistics.median(encode_ms), 2),
            "pairs_per_second": round(pairs / pair_seconds, 1),
        }
    else:
        reranker = CrossEncoderReranker(RERANKERS[label])
        for query in prepared.answerable:
            scores[query.query_id] = [
                round(s, 5) for s in reranker.score(query.question, [texts[c] for c in by_doc[query.doc_id]])
            ]
        timing = {"pairs_per_second": round(reranker.report.pairs_per_second, 2)}
    return {"scores": scores, "timing": {**timing, "load_average": os.getloadavg()[0]}, "model": RERANKERS[label]}


def rerank_key(label, slice_name):
    """Name every input that fixes one reranker's scores on one slice."""
    return CacheKey(
        "ri1", "rerank-scores", RERANKERS[label].split("/")[-1], {**slice_config(slice_name), "candidates": "contract"}
    )


rerank_scores = {
    (label, name): measured_result(
        rerank_key(label, name), lambda label=label, name=name: score_every_chunk(label, name)
    )
    for name in RERANK_SLICES
    for label in RERANKERS
}


def rerank(first_stage, label, slice_name, depth):
    """Reorder each first-stage top `depth` by a reranker's cached scores."""
    by_doc = contract_chunks(slice_name)
    cached = rerank_scores[(label, slice_name)]["scores"]
    doc_of = {query.query_id: query.doc_id for query in SLICES[slice_name][1].answerable}
    reranked = {}
    for q, ranked in first_stage.items():
        lookup = dict(zip(by_doc[doc_of[q]], cached[q]))
        head = sorted(ranked[:depth], key=lambda c: (-lookup[c], c))
        reranked[q] = head + [c for c in ranked[depth:] if c not in head]
    return reranked


print("cached reranker throughput on 8 CPUs, from the measurement run")
for (label, name), measured in rerank_scores.items():
    if name == "CUAD 20":
        print(f"  {label:<20}{measured['timing']['pairs_per_second']:>8} pairs per second")

cached reranker throughput on 8 CPUs, from the measurement run
  MiniLM-L6              20.09 pairs per second
  bge-reranker-v2-m3      0.82 pairs per second
  ColBERT MaxSim        3841.8 pairs per second


In [7]:
# Change these two to replay another first stage or depth from the cache, with no model.
FIRST_STAGES = {
    "SPLADE": lambda name: rankings[name]["splade"],
    "dense + SPLADE (0.3)": lambda name: fuse(
        [rankings[name]["dense"], rankings[name]["splade"]], [0.3, 0.7], limit=DEPTH
    ),
    "hybrid BM25 + dense (0.2)": lambda name: fuse(
        [rankings[name]["dense"], rankings[name]["bm25"]], [0.2, 0.8], limit=DEPTH
    ),
}
RERANK_DEPTHS = (10, 20, 30)

print(
    f"{'first stage':<27}{'reranker':<20}{'depth':>6}"
    + "".join(f"{name + ' fail':>22}{'MRR':>8}" for name in RERANK_SLICES)
)
rerank_rows = []
for stage, make in FIRST_STAGES.items():
    first = {name: make(name) for name in RERANK_SLICES}
    base = "".join(
        f"{score(first[name], name, SHALLOW_K[name])[0]:>22.4f}{score(first[name], name)[1]:>8.4f}"
        for name in RERANK_SLICES
    )
    print(f"{stage:<27}{'none':<20}{'':>6}{base}")
    for label in RERANKERS:
        for depth in RERANK_DEPTHS:
            cells = ""
            for name in RERANK_SLICES:
                failure, mrr = score(rerank(first[name], label, name, depth), name, SHALLOW_K[name])
                cells += f"{failure:>22.4f}{mrr:>8.4f}"
                rerank_rows.append((stage, label, depth, name, failure, mrr))
            print(f"{'':<27}{label:<20}{depth:>6}{cells}")

first stage                reranker             depth          CUAD 20 fail     MRR   ContractNLI 20 fail     MRR
SPLADE                     none                                      0.2000  0.6139                0.0838  0.8180
                           MiniLM-L6               10                0.2000  0.5160                0.0950  0.8049
                           MiniLM-L6               20                0.2080  0.5001                0.0894  0.8052
                           MiniLM-L6               30                0.2160  0.4979                0.0894  0.8052
                           bge-reranker-v2-m3      10                0.2000  0.6061                0.0782  0.8482
                           bge-reranker-v2-m3      20                0.1800  0.5951                0.0782  0.8502
                           bge-reranker-v2-m3      30                0.1800  0.5943                0.0782  0.8502
                           ColBERT MaxSim          10                0.2000  0.5378     

                           ColBERT MaxSim          20                0.2320  0.5360                0.0894  0.7706
                           ColBERT MaxSim          30                0.2320  0.5349                0.0894  0.7706


`bge-reranker-v2-m3` is the only reranker that lowers failure on CUAD 20, from 0.2000 to 0.1800 at depth 20 on SPLADE, and it lowers SPLADE's MRR as it does so, from 0.6139 to 0.5951. On ContractNLI 20 it improves both, failure at 3 from 0.0838 to 0.0782 and MRR from 0.8180 to 0.8502, the best ContractNLI figures measured.

MiniLM-L6 and ColBERT rescoring help only a weak first stage. On the BM25 and dense hybrid, MiniLM raises CUAD MRR from 0.4967 to 0.5110 and ColBERT to 0.5360. On SPLADE, both lower it, to 0.5001 and 0.5392, and the reranked order approaches what each reranker would produce over the whole contract. A reranker replaces the first stage's ordering with its own, so it pays only when its own ordering is better. MiniLM was trained on web search queries, which do not look like CUAD's clause-category questions.

For MiniLM to win here, its ordering would have to beat SPLADE's on contract language. The cheapest test is fine-tuning it on the CUAD training split's question and clause pairs, then replaying this cell with the new scores. For ColBERT rescoring to win, the first stage would have to miss what ColBERT ranks well, which a deeper unscoped collection would test.

## 7. What each lever costs, and which one wins

The first cell prints latency and index size on CUAD 20. The host was shared with other jobs during the cached run, at load averages of 32 to 50 on 32 CPUs for the search arms, and `scripts/measure_cpu_latency.py` run on the same host just after reported a 2.8-times speed-up from 1 thread to 8, against 5.1 on an idle host. The `bge-reranker-v2-m3` scores were computed earlier under heavier load, at 0.82 pairs per second, and the script measured 1.32 on the quieter host. The measured milliseconds are therefore higher than the bootcamp container gives, and their order is the reliable part. Read the idle-host figures beside them: SPLADE 57 ms end to end, ColBERT 24 ms, `bge-reranker-v2-m3` 1.6 pairs per second, and MiniLM-L6 30 pairs per second.

Index sizes do not depend on the host. ColBERT's token vectors take 72.8 MiB for 954 chunks, 52 times dense's 1.40 MiB, and 366 MiB on the full split. BM25 and SPLADE sparse vectors are 0.71 MiB and 1.44 MiB.

The second cell puts every lever in one table, with the rerankers at depth 20 on the `dense + SPLADE (0.3)` first stage, and names the winner of each column.

In [8]:
# Per-query latency on 8 CPUs. Search arms were timed end to end (query encoding
# included) over 50 CUAD 20 questions. A reranker adds depth / pairs-per-second.
# Figures measured with scripts/measure_cpu_latency.py on an idle machine are
# printed beside them where they cover the arm.
IDLE_HOST_MS = {"splade": 57.0, "colbert": 24.0}
IDLE_HOST_PAIRS_PER_SECOND = {"MiniLM-L6": 30.0, "bge-reranker-v2-m3": 1.6}

cuad_costs = arm_results["CUAD 20"]
latency = {arm: cuad_costs[arm]["latency"]["median_ms"] for arm in ("bm25", "dense", "splade", "colbert")}
size_mib = {arm: cuad_costs[arm]["index_bytes"] / 2**20 for arm in ("bm25", "dense", "splade", "colbert")}
loads = sorted({round(cuad_costs[arm]["latency"].get("load_average", 0)) for arm in ("dense", "splade", "colbert")})
print(f"CUAD 20 costs (954 chunks); host load average during the cached run: {loads}")
print(f"{'arm':<10}{'ms per query':>14}{'idle-host ms':>13}{'index MiB':>11}")
for arm, ms in latency.items():
    idle = f"{IDLE_HOST_MS[arm]:.0f}" if arm in IDLE_HOST_MS else "-"
    print(f"{arm:<10}{ms:>14.1f}{idle:>13}{size_mib[arm]:>11.2f}")
print()
print(f"{'reranker':<20}{'pairs/s here':>13}{'idle host':>10}{'ms at depth 20':>16}")
for label in RERANKERS:
    measured = rerank_scores[(label, "CUAD 20")]["timing"]
    added = 1000 * 20 / measured["pairs_per_second"] + measured.get("query_encode_ms", 0.0)
    idle = f"{IDLE_HOST_PAIRS_PER_SECOND[label]:.1f}" if label in IDLE_HOST_PAIRS_PER_SECOND else "-"
    print(f"{label:<20}{measured['pairs_per_second']:>13}{idle:>10}{added:>16.0f}")

CUAD 20 costs (954 chunks); host load average during the cached run: [32, 45, 50]
arm         ms per query idle-host ms  index MiB
bm25                 0.5            -       0.71
dense               85.0            -       1.40
splade             108.9           57       1.44
colbert            133.8           24      72.80

reranker             pairs/s here idle host  ms at depth 20
MiniLM-L6                   20.09      30.0             996
bge-reranker-v2-m3           0.82       1.6           24390
ColBERT MaxSim             3841.8         -              18


In [9]:
def lever_row(label, runs_by_slice, ms, mib):
    """Return one lever's scores on both reranked slices and its costs."""
    cells = [score(runs_by_slice[name], name, SHALLOW_K[name]) for name in RERANK_SLICES]
    return (label, *cells[0], *cells[1], ms, mib)


minilm = IDLE_HOST_PAIRS_PER_SECOND["MiniLM-L6"]
fused_ms = latency["dense"] + latency["splade"]
levers = [
    lever_row("BM25", {n: rankings[n]["bm25"] for n in RERANK_SLICES}, latency["bm25"], size_mib["bm25"]),
    lever_row("dense", {n: rankings[n]["dense"] for n in RERANK_SLICES}, latency["dense"], size_mib["dense"]),
    lever_row(
        "hybrid BM25 + dense (0.2)",
        {n: FIRST_STAGES["hybrid BM25 + dense (0.2)"](n) for n in RERANK_SLICES},
        latency["bm25"] + latency["dense"],
        size_mib["bm25"] + size_mib["dense"],
    ),
    lever_row("SPLADE", {n: rankings[n]["splade"] for n in RERANK_SLICES}, latency["splade"], size_mib["splade"]),
    lever_row("ColBERT", {n: rankings[n]["colbert"] for n in RERANK_SLICES}, latency["colbert"], size_mib["colbert"]),
    lever_row(
        "dense + SPLADE (0.3)",
        {n: FIRST_STAGES["dense + SPLADE (0.3)"](n) for n in RERANK_SLICES},
        fused_ms,
        size_mib["dense"] + size_mib["splade"],
    ),
]
for label in RERANKERS:
    pairs_per_second = rerank_scores[(label, "CUAD 20")]["timing"]["pairs_per_second"]
    extra = 1000 * 20 / pairs_per_second + rerank_scores[(label, "CUAD 20")]["timing"].get("query_encode_ms", 0.0)
    extra_mib = size_mib["colbert"] if label == "ColBERT MaxSim" else 0.0
    runs = {n: rerank(FIRST_STAGES["dense + SPLADE (0.3)"](n), label, n, 20) for n in RERANK_SLICES}
    levers.append(
        lever_row(
            f"dense + SPLADE, {label} top 20",
            runs,
            fused_ms + extra,
            size_mib["dense"] + size_mib["splade"] + extra_mib,
        )
    )

print(f"{'lever':<42}{'CUAD fail@10':>13}{'MRR':>8}{'NLI fail@3':>12}{'MRR':>8}{'ms/query':>10}{'index MiB':>11}")
for label, cuad_fail, cuad_mrr, nli_fail, nli_mrr, ms, mib in levers:
    print(f"{label:<42}{cuad_fail:>13.4f}{cuad_mrr:>8.4f}{nli_fail:>12.4f}{nli_mrr:>8.4f}{ms:>10.0f}{mib:>11.2f}")
print()
columns = {
    "CUAD failure@10": (1, min),
    "CUAD MRR": (2, max),
    "ContractNLI failure@3": (3, min),
    "ContractNLI MRR": (4, max),
    "latency": (5, min),
    "index size": (6, min),
}
for metric, (column, pick) in columns.items():
    best = pick(row[column] for row in levers)
    winners = [row[0] for row in levers if row[column] == best]
    print(
        f"best {metric:<22} {', '.join(winners)} ({best:.4f})"
        if column < 5
        else f"best {metric:<22} {', '.join(winners)}"
    )

lever                                      CUAD fail@10     MRR  NLI fail@3     MRR  ms/query  index MiB
BM25                                             0.2600  0.4528      0.1508  0.7395         0       0.71
dense                                            0.2400  0.4632      0.1732  0.6893        85       1.40
hybrid BM25 + dense (0.2)                        0.2320  0.4967      0.1397  0.7533        85       2.10
SPLADE                                           0.2000  0.6139      0.0838  0.8180       109       1.44
ColBERT                                          0.2240  0.5348      0.0894  0.7706       134      72.80
dense + SPLADE (0.3)                             0.1840  0.5942      0.0894  0.8084       194       2.84
dense + SPLADE, MiniLM-L6 top 20                 0.2120  0.5011      0.0894  0.8052      1189       2.84
dense + SPLADE, bge-reranker-v2-m3 top 20        0.1800  0.5990      0.0782  0.8502     24584       2.84
dense + SPLADE, ColBERT MaxSim top 20            0.2200

Each metric has its own winner, and each win holds on the slices named here, with every question scoped to its contract:

- Lowest failure on CUAD 20: `dense + SPLADE` fused at 0.3, reranked by `bge-reranker-v2-m3` at depth 20, at 0.1800. The same fusion without the reranker reaches 0.1840, one question more, at about a hundredth of the latency.
- Highest MRR on CUAD 20: SPLADE alone, at 0.6139. Fusion and every reranker lower it. On CUAD all, where no reranker ran, SPLADE alone also has the highest MRR (0.5687), and its failure (0.2066) is within two questions of the best fusion.
- Lowest failure at 3 and highest MRR on ContractNLI 20: `bge-reranker-v2-m3` at depth 20 on any of the three first stages, at 0.0782 failure and MRR from 0.8502 to 0.8508.
- Lowest latency and smallest index: BM25, at under a millisecond and 0.71 MiB, which it pays for with the highest CUAD 20 failure of the first stages.

For a CPU deployment on contracts, SPLADE alone gives the best ranking at about a tenth of a second, and adding dense fusion buys a few questions of recall. The cross-encoder is a recall lever for an offline or GPU path: at the idle-host 1.6 pairs per second, 20 candidates take about 12 seconds per question.

## 8. Exact search, HNSW, and quantization on a Qdrant server

Every dense arm above searched the package's Qdrant store in process, which scans exactly. HNSW graphs and quantized vectors exist only on a Qdrant server, so this section needs one. Without `QDRANT_URL` set, the cells print the committed measurement, which ran against a local Qdrant 1.19.1 release binary started without Docker as `docs/vector-store.md` describes. With `QDRANT_URL` set, the second cell re-measures live, loading the CUAD 20 chunk and question vectors from the cached `vector-collection` artifacts, so it needs no embedding model. Set `QDRANT_STORAGE_PATH` to the server's storage directory as well if you want the measured compression ratio.

Seven settings run twice: scoped to the question's contract, as everywhere else, and unscoped over all 954 chunks. The settings are an exact scan, HNSW with `hnsw_ef=16`, TurboQuant at 2 bits with rescoring, and TurboQuant and binary quantization at 1 bit, each with 4 times oversampling and rescoring and without rescoring. `QdrantSettings` lowers the per-segment indexing and full-scan thresholds, so the graph and the quantized vectors are built and used at this size. The last column is the search path the server's telemetry recorded, which confirms it. Agreement is the share of the exact top 10 that a setting also returns, and top-1 is the share of questions whose first result matches the exact one.

In [10]:
if RUN_EXPENSIVE:
    from aieng.advanced_rag.retrieval.embedders import SentenceTransformerEmbedder

QDRANT_URL = os.environ.get("QDRANT_URL")
QDRANT_VERSION = "1.19.1"
SERVER_SLICE = "CUAD 20"

SERVER_ARMS = {
    "exact": {},
    "HNSW, hnsw_ef=16": {"hnsw_ef": 16},
    "TurboQuant 2-bit, rescore": {"quantization": "turbo", "bits": 2, "rescore": True},
    "TurboQuant 1-bit, oversampling 4, rescore": {
        "quantization": "turbo",
        "bits": 1,
        "oversampling": 4,
        "rescore": True,
    },
    "TurboQuant 1-bit, no rescore": {"quantization": "turbo", "bits": 1, "rescore": False},
    "binary 1-bit, oversampling 4, rescore": {"quantization": "binary", "bits": 1, "oversampling": 4, "rescore": True},
    "binary 1-bit, no rescore": {"quantization": "binary", "bits": 1, "rescore": False},
}


def collection_key(kind):
    """Name what fixes the vectors: corpus, chunker, model, and prompt."""
    return CacheKey(
        "ri1",
        "vector-collection",
        "bge-small-en-v1.5",
        {**slice_config(SERVER_SLICE), "model": DENSE_MODEL, "vectors": kind},
    )


def load_vectors():
    """Return chunk and question vectors from the cache, embedding on a miss."""
    prepared = SLICES[SERVER_SLICE][1]
    stores = {kind: QdrantStore(":memory:", collection=f"cached-{kind}") for kind in ("chunks", "questions")}
    if all(fetch_collection(cache, collection_key(kind), store) for kind, store in stores.items()):
        return stores["chunks"].vectors(), stores["questions"].vectors()
    if not RUN_EXPENSIVE:
        raise FileNotFoundError("Missing cached vectors; enable RUN_EXPENSIVE to embed them")
    embedder = SentenceTransformerEmbedder(DENSE_MODEL)
    stores["chunks"].add(
        [c.chunk_id for c in prepared.chunks],
        embedder.encode([c.indexed_text for c in prepared.chunks]),
        [c.doc_id for c in prepared.chunks],
    )
    stores["questions"].add(
        [q.query_id for q in prepared.answerable],
        embedder.encode_queries([q.question for q in prepared.answerable]),
        [q.doc_id for q in prepared.answerable],
    )
    for kind, store in stores.items():
        with tempfile.TemporaryDirectory() as scratch:
            exported = export_collection(
                store, Path(scratch) / kind, metadata={"model": DENSE_MODEL, "query_prompt": embedder.query_prompt}
            )
            store_directory(cache, collection_key(kind), exported, write_tier="global")
    return stores["chunks"].vectors(), stores["questions"].vectors()

In [11]:
def search_paths(name):
    """Return the search paths the server took for one collection."""
    with urllib.request.urlopen(f"{QDRANT_URL}/telemetry?details_level=6", timeout=30) as response:
        collections = json.load(response).get("result", {}).get("collections", {}).get("collections", [])
    paths = set()
    for entry in collections:
        if entry.get("id") == name:
            for segment in entry.get("shards", [{}])[0].get("local", {}).get("segments", []):
                for searches in segment.get("vector_index_searches") or []:
                    paths.update(
                        key for key, value in searches.items() if isinstance(value, dict) and value.get("count")
                    )
    return sorted(paths)


def quantized_ratio(name):
    """Divide float32 by quantized vector bytes, if the server storage is readable."""
    storage = os.environ.get("QDRANT_STORAGE_PATH")
    if not storage:
        return None
    root = Path(storage) / "collections" / name
    original = sum(p.stat().st_size for p in root.rglob("matrix.dat"))
    quantized = sum(p.stat().st_size for p in root.rglob("quantized.data"))
    return round(original / quantized, 1) if quantized else None


def measure_server():
    """Build one server collection per setting and score it scoped and unscoped."""
    prepared = SLICES[SERVER_SLICE][1]
    vectors, question_vectors = load_vectors()
    keys = [c.chunk_id for c in prepared.chunks]
    rows = {}
    for scoped in (True, False):
        exact = None
        for label, fields in SERVER_ARMS.items():
            name = f"ri1-levers-{len(rows)}"
            store = QdrantStore(QDRANT_URL, collection=name, settings=QdrantSettings(**fields))
            store.add(keys, vectors, [c.doc_id for c in prepared.chunks])
            scopes = [q.doc_id if scoped else None for q in prepared.answerable]
            for vector, scope in list(zip(question_vectors, scopes))[:10]:
                store.search(vector, K, scope=scope)
            runs, times = {}, []
            for query, vector, scope in zip(prepared.answerable, question_vectors, scopes):
                began = time.perf_counter()
                runs[query.query_id] = [key for key, _ in store.search(vector, K, scope=scope)]
                times.append(1000 * (time.perf_counter() - began))
            exact = exact or runs
            failure, mrr = score(runs, SERVER_SLICE)
            rows[f"{'scoped' if scoped else 'unscoped'} | {label}"] = {
                "failure": failure,
                "mrr": mrr,
                "agreement_at_10": statistics.mean(len(set(runs[q]) & set(exact[q])) / len(exact[q]) for q in runs),
                "top1_match": statistics.mean(float(runs[q][:1] == exact[q][:1]) for q in runs),
                "median_ms": round(statistics.median(times), 2),
                "ratio": quantized_ratio(name),
                "search_paths": search_paths(name),
            }
            store.client.delete_collection(name)
    return {"rows": rows, "server": QDRANT_VERSION, "load_average": os.getloadavg()[0]}


server_key = CacheKey(
    "ri1",
    "qdrant-server",
    f"qdrant-{QDRANT_VERSION}",
    {**slice_config(SERVER_SLICE), "model": DENSE_MODEL, "arms": SERVER_ARMS},
)
if QDRANT_URL:
    server = cache.load_or_compute(
        server_key, measure_server, force_refresh=True, write_tier="global" if RUN_EXPENSIVE else "local"
    ).value
    server_source = f"measured live against {QDRANT_URL}"
else:
    cached = cache.get(server_key)
    if cached is None:
        raise FileNotFoundError("No cached server measurement; set QDRANT_URL to a running Qdrant server")
    server = cached.value
    server_source = f"committed measurement from a local Qdrant {server['server']} release binary"

print(f"CUAD 20, 954 chunks, 250 answerable questions, top 10 ({server_source})")
print(f"{'search':<9}{'setting':<44}{'ratio':>6}{'agree@10':>10}{'top-1':>7}{'fail':>8}{'MRR':>8}{'ms':>7}  path")
for label, row in server["rows"].items():
    scope, setting = label.split(" | ")
    ratio = f"{row['ratio']:.1f}" if row["ratio"] else "1.0" if "exact" in setting or "HNSW" in setting else "-"
    print(
        f"{scope:<9}{setting:<44}{ratio:>6}{row['agreement_at_10']:>10.3f}{row['top1_match']:>7.3f}"
        f"{row['failure']:>8.4f}{row['mrr']:>8.4f}{row['median_ms']:>7.1f}  {', '.join(row['search_paths'])}"
    )

CUAD 20, 954 chunks, 250 answerable questions, top 10 (committed measurement from a local Qdrant 1.19.1 release binary)
search   setting                                      ratio  agree@10  top-1    fail     MRR     ms  path
scoped   exact                                          1.0     1.000  1.000  0.2400  0.4632    6.7  filtered_plain
scoped   HNSW, hnsw_ef=16                               1.0     0.998  1.000  0.2400  0.4633    6.4  filtered_large_cardinality, filtered_small_cardinality
scoped   TurboQuant 2-bit, rescore                     14.8     0.997  1.000  0.2400  0.4632    7.0  filtered_small_cardinality
scoped   TurboQuant 1-bit, oversampling 4, rescore     27.4     1.000  1.000  0.2400  0.4632    5.4  filtered_small_cardinality
scoped   TurboQuant 1-bit, no rescore                  27.4     0.720  0.580  0.2320  0.4495    7.2  filtered_small_cardinality
scoped   binary 1-bit, oversampling 4, rescore         32.0     0.994  0.996  0.2400  0.4636    6.0  filtered_small_ca

With rescoring, compression costs nothing measurable here. TurboQuant at 2 bits stores the vectors at 14.8 times smaller and returns 99.7% of the exact top 10 with the same first result on every question. TurboQuant at 1 bit, at 27.4 times, returns all of it with 4 times oversampling and rescoring. HNSW at `hnsw_ef=16` returns 99.8% scoped and 99.5% unscoped, and the failure rate and MRR move by at most one question. Median latency stayed between 4.7 and 7.6 ms with no ordering by setting, because 1.4 MB of vectors fit in memory whichever way they are searched, and most of each millisecond is the HTTP round trip.

Without rescoring, the damage shows in agreement and the first result before it shows in the task metric. TurboQuant at 1 bit keeps 72.0% of the exact top 10 and the same first result on 58.0% of questions, and its failure rate falls from 0.2400 to 0.2320, two questions in the other direction. Binary at 1 bit keeps 64.4% and its failure rises to 0.2760. A report that showed only failure would rank the unrescored TurboQuant arm above exact search. Report agreement with the exact ranking whenever you change how vectors are stored.

The unscoped failure rate, 0.7720 for exact search, is a property of CUAD, not of the store: the 41 question templates are the same for every contract, so a question with no scope cannot say which contract it means. The unscoped rows are there to put the HNSW graph on its unfiltered path; read their agreement columns.

What would have to be true for HNSW and quantization to win on latency or memory: a collection large enough that a scan costs more than walking a graph, and vectors that no longer fit in RAM. On 23,525 ServerFault chunks, `hnsw_ef=16` measured 7.7 ms against 10.3 ms exact, within the noise, and quantization saved no measurable memory at 36 MB of vectors (`docs/vector-store.md`). The cheapest test is the same cell on a collection of a few hundred thousand points, such as a chunked public corpus embedded once on a GPU and shipped as a `vector-collection` artifact.

## 9. Using these results on your own corpus

The cached rankings make every comparison above a replay, so the useful exercise is to change one lever and read the table again: a fusion weight in section 4, a first stage or a depth in section 6. To measure your own documents, build a `Slice` as notebook 01 does, add it to `SLICES`, and enable `RUN_EXPENSIVE`. The cache keys name the corpus digest and the chunker, so a new slice never reads another slice's results.

For a comparison that holds up:

- Choose a fusion weight on the largest question set you have, and report the sweep rather than one weight.
- Report failure and MRR together. On these contracts the lever with the lowest failure and the lever with the highest MRR are often different.
- Report latency with the host it was measured on, and the load on that host.
- When a reranker loses, check whether the first stage was already ranking well. A reranker trained on other text can undo a good first stage.
- When you change how vectors are stored, report agreement with the exact ranking as well as the task metrics.

Notebook 01 explains the harness, and [GUIDE.md](GUIDE.md) maps the rest of the repository. The vector store's settings and their earlier measurements are in [`docs/vector-store.md`](../../docs/vector-store.md).